# Семинар 2. Исследование методов линейной регрессии

#### Критерий оценивания:
#### Пункт 14: максимум 0,5 балла
#### Пункт 15: максимум 0,5 балла

#### Итого за работу: максимум 1 балл
#### P.S. пункты 14 и 15 без пунктов 1-13 не засчитываются, ответы на вопросы из ИИ не засчитываются

1. Загрузите датасет и выведите на экран первые несколько строк

In [48]:
import pandas as pd

# Загрузка датасета
data = pd.read_csv('auto_dataset.csv')

# Вывод первых строк датасета
data.head()

,brand,model,vehicleType,gearbox,fuelType,notRepairedDamage,powerPS,kilometer,autoAgeMonths,price
0,volkswagen,golf,kleinwagen,manuell,benzin,nein,75,150000,177,1500
1,skoda,fabia,kleinwagen,manuell,diesel,nein,69,90000,93,3600
2,bmw,3er,limousine,manuell,benzin,ja,102,150000,246,650
3,peugeot,2_reihe,cabrio,manuell,benzin,nein,109,150000,140,2200
4,mazda,3_reihe,limousine,manuell,benzin,nein,105,150000,136,2000


2. Разбейте выборку на признаки и ответы. Закодируйте категориальные признаки.

In [49]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HIGH_CARD = ['brand', 'model']
LOW_CARD  = ['vehicleType', 'gearbox', 'fuelType', 'notRepairedDamage']
NUM_COLS  = ['powerPS', 'kilometer', 'autoAgeMonths']

X, y = data.iloc[0:, 0:-1].copy(), np.array(data.iloc[0:, -1:]).ravel()
y = np.asarray(y, dtype=float)

# --- target encoding (по всей выборке — как требует порядок задания) ---
global_mean = y.mean()

for col in HIGH_CARD:
    means = pd.Series(y, index=X[col].values).groupby(level=0).mean()
    X[col] = X[col].map(means).fillna(global_mean)

# --- One-Hot для LOW_CARD ---
ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
ohe.fit(X[LOW_CARD])

ohe_cols = list(ohe.get_feature_names_out(LOW_CARD))

def apply_ohe(df):
    arr = ohe.transform(df[LOW_CARD])
    ohe_df = pd.DataFrame(arr, columns=ohe_cols, index=df.index)
    return pd.concat([df.drop(columns=LOW_CARD), ohe_df], axis=1)

X = apply_ohe(X)

# --- числовые колонки ---
num_used = NUM_COLS + HIGH_CARD

scaler = StandardScaler().fit(X[num_used].astype(float))
X[num_used] = scaler.transform(X[num_used].astype(float))

X.head()

,brand,model,powerPS,kilometer,autoAgeMonths,vehicleType_bus,vehicleType_cabrio,vehicleType_coupe,vehicleType_kleinwagen,vehicleType_kombi,vehicleType_limousine,vehicleType_suv,gearbox_manuell,fuelType_benzin,fuelType_diesel,fuelType_hybrid,fuelType_lpg,notRepairedDamage_nein
0,-0.232965,-0.405164,-0.965578,0.667505,0.416437,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,0.037622,-0.116795,-1.063661,-0.839845,-0.679725,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.691668,-0.172015,-0.524204,0.667505,1.316856,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,-0.575409,-0.536062,-0.409774,0.667505,-0.066396,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
4,-0.697748,-0.670080,-0.475163,0.667505,-0.118594,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0


3. Разбейте датасет на train val test в отношении 8:1:1

In [50]:
X_train, y_train = X.iloc[0:800],  y[0:800]
X_val,   y_val   = X.iloc[800:900], y[800:900]
X_test,  y_test  = X.iloc[900:],    y[900:]
results = []

4. Исследуйте VGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [51]:
lyamdas = [10 ** -5, 10 ** -4, 10 ** -3, 10 ** -2, 10 ** -1, 1]
MSE_train, R2_train, MSE_val = [], [], []

l = len(X_train)
epsilon = 0.0001
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(X_train.shape[1])
    MSE_val_lyamda = []
    for k in range(1200):
        grad = (2 / l) * X_train.T @ (X_train @ w - y_train)
        if np.linalg.norm(grad) < epsilon:
            break
        step = lyamda
        # step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * grad

        y_pred_val = X_val @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    y_pred_train = X_train @ w
    
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)
    
print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))
R2_train

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'VGD (const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1200,
})

[109421121.37188633, 55021692.690228656, 17968790.096813723, 17750432.960446004, 17692555.185671333, nan]
17692555.185671333
Best lyamda = 0.1
mse_test = 17603380.941569068
r2_test = 0.7636334763888594


5. Исследуйте VGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите VGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте VGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте VGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
lyamdas = [10 ** -5, 10 ** -4, 10 ** -3, 10 ** -2, 10 ** -1, 1]
MSE_train, R2_train, MSE_val = [], [], []

l = len(X_train)
epsilon = 0.0001
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(X_train.shape[1])
    MSE_val_lyamda = []
    for k in range(1200):
        grad = (2 / l) * X_train.T @ (X_train @ w - y_train)
        if np.linalg.norm(grad) < epsilon:
            break
        # step = lyamda
        step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * grad

        y_pred_val = X_val @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    y_pred_train = X_train @ w
    
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)
    
print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))
R2_train

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'VGD (not const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1200,
})


6. Исследуйте SGD с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []

l = len(X_train)
batch_size = 32
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(X_train.shape[1])
    indices = np.random.permutation(len(X_train))  # перемешали
    MSE_val_lyamda = []
    for k in range(1000):
        # выбираем индексы батча
        batch_idx = indices[k * batch_size : (k + 1) * batch_size]
        if len(batch_idx) == 0:
            indices = np.random.permutation(len(X_train))
            batch_idx = indices[:batch_size]

        X_b = X_train.iloc[batch_idx]
        y_b = y_train[batch_idx]

        grad = (2 / len(batch_idx)) * X_b.T @ (X_b @ w - y_b)
        step = lyamda
        # step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * grad

        y_pred_val = X_val @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    y_pred_train = X_train @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)
    
    
print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'SGD (const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[111081049.22580482, 61870583.47296848, 17940106.01342611, 17214498.169302665, 19024391.679018337, nan]
17214498.169302665
Best lyamda = 0.1
mse_test = 18763868.692839276
r2_test = 0.7480512165734559


7. Исследуйте SGD с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SGD на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SGD на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SGD c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []

l = len(X_train)
batch_size = 32
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(X_train.shape[1])
    indices = np.random.permutation(len(X_train))  # перемешали
    MSE_val_lyamda = []
    for k in range(1000):
        # выбираем индексы батча
        batch_idx = indices[k * batch_size : (k + 1) * batch_size]
        if len(batch_idx) == 0:
            indices = np.random.permutation(len(X_train))
            batch_idx = indices[:batch_size]

        X_b = X_train.iloc[batch_idx]
        y_b = y_train[batch_idx]

        grad = (2 / len(batch_idx)) * X_b.T @ (X_b @ w - y_b)
        # step = lyamda
        step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * grad

        y_pred_val = X_val @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    y_pred_train = X_train @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)
    
    
print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'SGD (not const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[118342291.28058337, 113900325.64526531, 77905301.41078998, 18040105.461660568, 17618267.383513257, 17023963.37243975]
17023963.37243975
Best lyamda = 1
mse_test = 17202023.389811613
r2_test = 0.7690226393882218


8. Исследуйте SAG с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train['bias'] = 1.0
X_val['bias']   = 1.0
X_test['bias']  = 1.0

MSE_train, R2_train, MSE_val = [], [], []
d = X_train.shape[1]            # число признаков
l = len(X_train)
best_lyamda = False


for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    g_memory = np.zeros((l, d))    # сохранённые градиенты объектов
    g_avg = np.zeros(d)            # средний градиент g_k
    MSE_val_lyamda = []
    
    for k in range(10000):
        j = np.random.randint(l)                             # случайный объект

        g_old = g_memory[j].copy()        # старый g_j
        g_new = 2 * (X_train.iloc[j].values.T @ w - y_train[j]) * X_train.iloc[j].values        # новый g_j

        g_memory[j] = g_new                                  # записали в память
        g_avg = g_avg + (g_new - g_old) / l                 # обновили среднее

        step = lyamda
        # step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * g_avg

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'SAG (const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[np.float64(38703814.40637229), np.float64(18008044.81416327), np.float64(18152210.25384133), np.float64(17880150.281001624), np.float64(1230754171583.1687), np.float64(7.199068091233025e+67)]
17880150.281001624
Best lyamda = 0.01
mse_test = 17760121.231079217
r2_test = 0.7615288717413604


9. Исследуйте SAG с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите SAG на train
* найдите и запомните R^2_train и Loss_train
* протестируйте SAG на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте SAG c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
X_train = X_train.copy()
X_val   = X_val.copy()
X_test  = X_test.copy()

X_train['bias'] = 1.0
X_val['bias']   = 1.0
X_test['bias']  = 1.0

MSE_train, R2_train, MSE_val = [], [], []
d = X_train.shape[1]            # число признаков
l = len(X_train)
best_lyamda = False


for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    g_memory = np.zeros((l, d))    # сохранённые градиенты объектов
    g_avg = np.zeros(d)            # средний градиент g_k
    MSE_val_lyamda = []
    
    for k in range(10000):
        j = np.random.randint(l)                             # случайный объект

        g_old = g_memory[j].copy()        # старый g_j
        g_new = 2 * (X_train.iloc[j].values.T @ w - y_train[j]) * X_train.iloc[j].values        # новый g_j

        g_memory[j] = g_new                                  # записали в память
        g_avg = g_avg + (g_new - g_old) / l                 # обновили среднее

        # step = lyamda
        step = lyamda * (1 / (1 + k)) ** 0.5
        w = w - step * g_avg

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'SAG (not const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[np.float64(65197243.7168617), np.float64(59964710.623655364), np.float64(29585063.628288835), np.float64(17707903.416507013), np.float64(18229273.076324403), np.float64(17983396.35531585)]
17707903.416507013
Best lyamda = 1
mse_test = 17635530.371764258
r2_test = 0.7632017951637277


10. Исследуйте Momentum с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []
# d = X_train.shape[1]            # число признаков
# l = len(X_train)
best_lyamda = False


for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    h = 0
    MSE_val_lyamda = []
    
    for k in range(1000):
        grad = 2 / l * X_train.T @ (X_train @ w - y_train)

        step = lyamda
        # step = lyamda * (1 / (1 + k)) ** 0.5

        h = 0.9 * h + step * grad
        w = w - h

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'Momentum (const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning: overflow encountered in scalar add
  MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning: overflow encountered in square
  MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning: overflow encountered in square
  MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning: overflow encountered in square
  MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning: overflow encountered in square
  MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
C:\Users\Anna\AppData\Local\Temp\ipykernel_12952\1475686597.py:23: RuntimeWarning

[np.float64(37815400.91714629), np.float64(17829031.25456062), np.float64(18134451.64777785), np.float64(17931801.460281953), np.float64(18008683.748116545), np.float64(nan)]
17829031.25456062
Best lyamda = 0.1
mse_test = 17753726.68753728
r2_test = 0.7616147334307771


11. Исследуйте Momentum с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Momentum на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Momentum на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Momentum c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []
# d = X_train.shape[1]            # число признаков
# l = len(X_train)
best_lyamda = False


for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    h = 0
    MSE_val_lyamda = []
    
    for k in range(1000):
        grad = 2 / l * X_train.T @ (X_train @ w - y_train)

        # step = lyamda
        step = lyamda * (1 / (1 + k)) ** 0.5

        h = 0.9 * h + step * grad
        w = w - h

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'Momentum (not const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[np.float64(63315866.95128355), np.float64(45807714.18854247), np.float64(17794823.64688312), np.float64(18044746.83152634), np.float64(17927678.590562128), np.float64(17993079.679705)]
17794823.64688312
Best lyamda = 1
mse_test = 17741498.191059314
r2_test = 0.7617789295707733


12. Исследуйте Adam с постоянным шагом n:

переберите n в логарифмической сетке от 10^-5 до 1

для каждого n:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший n по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c лучшим n на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []
# d = X_train.shape[1]            # число признаков
# l = len(X_train)
epsilon = 10 ** -8
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    m, v = np.zeros(d), np.zeros(d)
    MSE_val_lyamda = []
    
    for k in range(1000):
        grad = 2 / l * X_train.T @ (X_train @ w - y_train)

        # step = lyamda * (1 / (1 + k)) ** 0.5
        step = lyamda
        m = 0.9 * m + (1 - 0.9) * grad
        v = 0.999 * v + (1 - 0.999) * grad ** 2
        m_hat = m / (1 - 0.9 ** (k + 1))
        v_hat = v / (1 - 0.999 ** (k + 1))
        w = w - step * (m_hat / (v_hat ** 0.5 + epsilon))

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    # if lyamda == 10 ** -2:
    #     best_weight = w
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'Adam (const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[np.float64(65801453.96881711), np.float64(65795818.721948475), np.float64(65739490.221786015), np.float64(65178562.021791354), np.float64(59785077.3028376), np.float64(28166250.14821712)]
28166250.14821712
Best lyamda = 1
mse_test = 33293110.02712368
r2_test = 0.5529622006457013


13. Исследуйте Adam с переменным шагом n(lyamda) по формуле TimeDecayLR (из лекции):

переберите lyamda в логарифмической сетке от 10^-5 до 1

для каждого lyamda:

* обучите Adam на train
* найдите и запомните R^2_train и Loss_train
* протестируйте Adam на val, запомните Loss_val

найдите наилучший lyamda по минимальному Loss_val, запомните его Loss_train, R^2_train, Loss_val;

протестируйте Adam c n(best_lyamda) на test, запомните Loss_test, R^2_test, число итераций на test.

In [ ]:
MSE_train, R2_train, MSE_val = [], [], []
# d = X_train.shape[1]            # число признаков
# l = len(X_train)
epsilon = 10 ** -8
best_lyamda = False

for lyamda in lyamdas:
    w = np.zeros(d)
    w[-1] = y_train.mean()
    m, v = np.zeros(d), np.zeros(d)
    MSE_val_lyamda = []
    
    for k in range(1000):
        grad = 2 / l * X_train.T @ (X_train @ w - y_train)

        step = lyamda * (1 / (1 + k)) ** 0.5
        # step = lyamda
        m = 0.9 * m + (1 - 0.9) * grad
        v = 0.999 * v + (1 - 0.999) * grad ** 2
        m_hat = m / (1 - 0.9 ** (k + 1))
        v_hat = v / (1 - 0.999 ** (k + 1))
        w = w - step * (m_hat / (v_hat ** 0.5 + epsilon))

        y_pred_val = X_val.values @ w
        MSE_val_lyamda.append(sum((y_val - y_pred_val) ** 2) / len(X_val))

    # if lyamda == 10 ** -2:
    #     best_weight = w
        
    y_pred_train = X_train.values @ w
    MSE_train.append(sum((y_train - y_pred_train) ** 2) / len(X_train))
    MSE_val.append(MSE_val_lyamda)

    if not best_lyamda:
        best_lyamda = lyamda
        best_weight = w
        err = MSE_val[-1][-1]
    elif err > MSE_val[-1][-1]:
        best_lyamda = lyamda
        best_weight = w

    # R²_train
    up_train = np.sum((y_train - y_pred_train) ** 2)
    down_train = np.sum((y_train - y_train.mean()) ** 2)
    r2_train = 1 - up_train / down_train
    R2_train.append(r2_train)


print([i[-1] for i in MSE_val])
print(min([i[-1] for i in MSE_val]))

print(f'Best lyamda = {best_lyamda}')

y_pred_test = X_test @ best_weight
mse_test = sum((y_test - y_pred_test) ** 2) / len(X_test)
print(f'mse_test = {mse_test}')

# R²_test
up_test = np.sum((y_test - y_pred_test) ** 2)
down_test = np.sum((y_test - y_test.mean()) ** 2)
r2_test = 1 - up_test / down_test

print(f'r2_test = {r2_test}')

results.append({
    'method': 'Adam (not const)',
    'best_step': best_lyamda,
    'Loss_train': MSE_train[lyamdas.index(best_lyamda)],
    'Loss_test': mse_test,
    'R2_train': R2_train[lyamdas.index(best_lyamda)],
    'R2_test': r2_test,
    'iters_test': 1000,
})

[np.float64(65802041.43649879), np.float64(65801693.15715231), np.float64(65798210.450905144), np.float64(65763392.10200743), np.float64(65416071.23194377), np.float64(62015565.87446134)]
62015565.87446134
Best lyamda = 1
mse_test = 71743535.45425998
r2_test = 0.036675390756809256


14. Постройте итоговую сравнительную таблицу со следующими столбцами:

1) название метода

2) значение лучшего шага (если n) или функция лучшего шага (если n(lyamda))

3) Loss_train

4) Loss_test

5) R^2 train

6) R^2 test

7) число итераций на test

In [ ]:
import pandas as pd

columns = [
    'method',
    'best_step',
    'Loss_train',
    'Loss_test',
    'R2_train',
    'R2_test',
    'iters_test',
]

df = pd.DataFrame(results, columns=columns)
df

,method,best_step,Loss_train,Loss_test,R2_train,R2_test,iters_test
0,VGD (const),0.10,1.648053e+07,1.760338e+07,0.720230,0.763633,1200
1,VGD (not const),1.00,1.649902e+07,1.763881e+07,0.719916,0.763158,1200
2,SGD (const),0.10,1.697867e+07,1.876387e+07,0.711774,0.748051,1000
3,SGD (not const),1.00,1.668993e+07,1.720202e+07,0.716675,0.769023,1000
4,SAG (const),0.01,1.636930e+07,1.776012e+07,0.722118,0.761529,1000
5,SAG (not const),1.00,1.636565e+07,1.763553e+07,0.722180,0.763202,1000
6,Momentum (const),0.10,1.635345e+07,1.775373e+07,0.722387,0.761615,1000
7,Momentum (not const),1.00,1.635419e+07,1.774150e+07,0.722375,0.761779,1000
8,Adam (const),1.00,2.896215e+07,3.329311e+07,0.508345,0.552962,1000
9,Adam (not const),1.00,5.604837e+07,7.174354e+07,0.048535,0.036675,1000


15. Сделайте вывод о том, какой метод и шаг линейной регрессии самый лучший для данной выборки и ответьте на вопросы:

1) почему именно этот метод и этот шаг самый лучший (по каким данным из таблицы вы сделали такой вывод)

2) расскажите простыми словами суть R^2_train и R^2_test?

3) как R^2_train и R^2_test помогают сравнивать методы? почему оба эти значения надо вычислять для данного выбора?

1) SGD с переменным шагом (λ = 1). У него самый высокий R² на тесте — 0.77, и при этом R² на train близок к нему (0.72), то есть модель не переобучена. Худший — Adam: слишком большой шаг, он разошёлся, R² на тесте упал до 0.55 и 0.04.

2) Первый показывает, насколько хорошо модель подогналась под обучающие данные. Второй — насколько хорошо она предсказывает новые данные, которых не видела.

3) По тесту выбирают лучший метод. По train смотрят, нет ли переобучения: если на train сильно выше, чем на тесте — модель зазубрила данные и плохо обобщает. У вас они близки, значит переобучения нет.